## 파이토치 실습

### 1. 파이토치로 선형 회귀 구현하기

#### 선형회귀 가설식

$$
H(x) = Wx+b
$$

> 4-2에서 NumPy로 직접 구현한 경사하강법을 파이토치로 옮기면 다음과 같이 대응됨

| 단계 | NumPy (4-2) | PyTorch |
|---|---|---|
| 모델 | `w * x + b` | `nn.Linear(1, 1)` |
| loss 계산 | `np.mean((y_pred - y) ** 2)` | `F.mse_loss(prediction, y_train)` |
| 기울기 계산 | `dw, db = gradient(x, y, w, b)` (직접 미분) | `optimizer.zero_grad()`, `loss.backward()` (자동 미분) |
| 업데이트 | `w = w - lr * dw` | `optimizer.step()` |

In [56]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(777)

In [ ]:
import numpy as np

# 4-2와 같은 데이터: 공부 시간(hours) → 점수(score)
rng = np.random.default_rng(0)
hours = np.round(rng.uniform(1, 10, 20), 1)
score = np.round(5 * hours + 40 + rng.normal(0, 5, 20), 1)  # 실제 관계: 점수 ≈ 5 x 시간 + 40 (+ 노이즈)

# (샘플 수, 특성 수) = (20, 1) 형태의 텐서로 변환
x_train = torch.FloatTensor(hours).view(-1, 1)
y_train = torch.FloatTensor(score).view(-1, 1)
print(x_train.shape, y_train.shape)

In [58]:
# 모델 선언 및 초기화
model = nn.Linear(1, 1) # (input_dim, output_dim)

In [59]:
# 모델의 초기화된 가중치와 편향 출력
print(list(model.parameters()))

[Parameter containing:
tensor([[-0.8362]], requires_grad=True), Parameter containing:
tensor([-0.0177], requires_grad=True)]


--------------------------
#### 1) 옵티마이저 - 경사 하강법(Gradient Descent)

* 손실 함수(Loss function): 일반적으로 평균 제곱 오차 (Mean Squared Error, MSE) 사용
* 손실 함수(Loss function) = 비용 함수(Cost function) = 오차 함수(Error function) = 목적 함수 (Objective function)

<center>

$L(w,b)= \sum_{i=1}^{n} (y^{(i)}-\hat{y}^{(i)})^2$

</center>


In [60]:
from IPython.display import Image
Image(url='https://raw.githubusercontent.com/Hyelim-Shin/AIdev/master/Images/10-1.GD.png', width=500)

$$
gradient = \frac{\partial L(W)}{\partial W}
$$

* 기울기가 음수일 때 (Negative gradient): W의 값이 증가

$$
W := W-\alpha \times (-gradient) = W + \alpha \times gradient
$$

* 기울기가 양수일 때 (Positive gradient): W의 값이 감소

$$
W := W-\alpha \times gradient
$$


In [61]:
# 다양한 최적화 알고리즘들
# [출처] https://ganghee-lee.tistory.com/24
Image(url='https://raw.githubusercontent.com/Hyelim-Shin/AIdev/master/Images/10-1.optimizer.png', width=500)

##### 자동 미분 (Autograd)

In [62]:
w = torch.tensor(2.0, requires_grad=True)

y = w**2
z = 2 * y + 5

In [63]:
z.backward()

In [64]:
w.grad

tensor(8.)

**계산 그래프**

* 파이토치는 유향 비순환 그래프(Directed Acyclic Graph, DAG)를 기반으로 계산을 수행
    - 노드(node): 변수(텐서 등)를 의미
    - 엣지(edge): 연산(곱하기, 더하기 등)을 의미

<center>

$z=2\times(a-b)+c$

</center>


In [65]:
# 파이토치 계산 그래프
Image(url='https://raw.githubusercontent.com/rickiepark/ml-with-pytorch/main/ch13/figures/13_01.png', width=400)

In [66]:
a = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(2.0, requires_grad=True)
c = torch.tensor(3.0, requires_grad=True)

z = 2 * (a - b) + c
print(z)

tensor(1., grad_fn=<AddBackward0>)


In [67]:
z.backward()
print(a.grad, b.grad, c.grad)  # dz/da = 2, dz/db = -2, dz/dc = 1

tensor(2.) tensor(-2.) tensor(1.)


* 미분값(`.grad`)은 `requires_grad=True`인 **leaf 텐서**에만 저장됨
    - leaf 텐서: 사용자가 직접 생성한 텐서 (ex. `w`, `a`, `b`, `c`, 모델의 파라미터)
    - non-leaf 텐서: 연산의 결과로 만들어진 텐서 (ex. `y`, `z`)
    - 중간 텐서(`y`)의 미분값도 역전파 과정에서 계산되지만, 메모리 절약을 위해 저장하지 않음 → `y.grad`는 `None`
    - 학습에서 업데이트할 대상은 파라미터(leaf)이므로 leaf의 미분값만 있으면 충분함

#### 2) 데이터 로드

전체 데이터를 하나의 행렬로 선언하여 전체 데이터에 대해서 경사 하강법을 수행하여 학습할 수 있음. 만약, 데이터가 수십만개 이상이라면 전체 데이터에 대해서 경사 하강법을 수행하는 것은 매우 느릴 뿐만 아니라 많은 계산량을 필요로 함. 따라서 전체 데이터를 더 작은 단위로 나누어서 해당 단위로 학습하는 방식을 사용함.

* 에포크(Epoch): 전체 훈련 데이터가 학습에 한 번 사용된 주기
* 미니배치(minibatch): 전체 데이터를 나누는 단위
* 배치 크기: 미니 배치의 크기
    - 보통 2의 제곱수 사용
* 이터레이션(Iteration): 한 번의 에포크 내에서 이루어지는 매개변수 $(W$와 $b)$ 업데이트 횟수

In [68]:
Image(url='https://raw.githubusercontent.com/Hyelim-Shin/AIdev/master/Images/10-1.data.png', width=500)

-----------------------

In [69]:
# optimizer 설정. 경사 하강법 SGD를 사용하고 학습률(learning rate, lr)은 0.01로 설정
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [ ]:
# 전체 훈련 데이터에 대해 경사 하강법을 3000회 반복 (4-2와 동일)

# 총 에포크 수 설정
n_epochs = 3000
for epoch in range(n_epochs + 1):
    # 1) 가설모델 H(x)=Wx+b 계산
    prediction = model(x_train)

    # 2) loss 계산
    loss = F.mse_loss(prediction, y_train)  # <-- 파이토치에서 제공하는 평균 제곱 오차 함수

    # 3) loss로 H(x) 개선
    # gradient를 0으로 초기화
    optimizer.zero_grad()
    # 손실 함수를 미분하여 gradient 계산
    loss.backward()
    # W, b 업데이트
    optimizer.step()

    if epoch % 300 == 0:
        print('Epoch {:4d}/{} Loss: {:.6f}'.format(
            epoch, n_epochs, loss.item()
        ))

In [ ]:
# 7시간 공부한 학생의 점수 예측
new_var = torch.FloatTensor([[7.0]])
# 입력한 값 7에 대해서 예측값 y를 리턴받아서 pred_y에 저장
pred_y = model(new_var) # forward 연산
# 점수 ≈ 5 x 시간 + 40 이므로 입력이 7이라면 75에 가까운 값이 나와야 제대로 학습이 된 것
print("훈련 후 7시간 공부했을 때의 예측 점수 :", pred_y)

In [ ]:
# 4-2의 경사하강법 결과(w ≈ 4.670, b ≈ 42.177)와 비교
print(list(model.parameters()))

* forward 연산: $H(x)$ 식에 입력 $x$로부터 예측된 $y$를 얻는 것
    - ex) 학습전 ```prediction = model(x_train)```, 학습후 ```pred_y = model(new_var)```
* backward 연산: 학습 과정에서 비용 함수를 미분하여 기울기를 구하는 것
    - ex) ```cost.backward()```

### 2. 파이토치로 로지스틱 회귀 구현하기

#### 로지스틱 회귀의 가설식

$$
H(x) = sigmoid(Wx+b)
$$

> 5-1에서 NumPy로 직접 구현한 로지스틱 회귀를 파이토치로 옮기면 다음과 같이 대응됨

| 단계 | NumPy (5-1) | PyTorch |
|---|---|---|
| 모델 | `sigmoid(w * x + b)` | `nn.Sequential(nn.Linear(1, 1), nn.Sigmoid())` |
| loss 계산 | `bce(y, p)` (직접 구현) | `F.binary_cross_entropy(hypothesis, y_train)` |
| 기울기 계산 | `dw = np.mean((y_pred - y) * x)` (직접 미분) | `loss.backward()` |
| 업데이트 | `w = w - lr * dw` | `optimizer.step()` |

In [ ]:
# 5-1과 같은 데이터: 성적(score) → 합격 여부(result), 60점 이상이면 합격
score = np.array([45, 50, 55, 60, 65, 70])
result = np.array([0, 0, 0, 1, 1, 1])

# 5-1과 같이 경사하강법이 잘 동작하도록 성적을 표준화
x = (score - score.mean()) / score.std()

x_train = torch.FloatTensor(x).view(-1, 1)
y_train = torch.FloatTensor(result).view(-1, 1)
print(x_train.shape, y_train.shape)

* ```nn.Sequential()```: 여러 함수들을 연결

In [ ]:
model = nn.Sequential(
    nn.Linear(1, 1), # input_dim=1, output_dim=1
    nn.Sigmoid() # 출력은 시그모이드 함수를 거친다
)

#### 1) 손실 함수(Loss Function)

**크로스 엔트로피 함수 (Cross Entropy Function)**


$$
L(W)=-\frac{1}{n} \sum_{i=1}^n \sum_{j=1}^k y_j log(p_j)
$$


In [ ]:
# optimizer 설정
optimizer = optim.SGD(model.parameters(), lr=0.5)  # 5-1과 동일

nb_epochs = 1000
for epoch in range(nb_epochs + 1):

    # H(x) 계산
    hypothesis = model(x_train)

    # cost 계산
    loss = F.binary_cross_entropy(hypothesis, y_train)

    # cost로 H(x) 개선
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # 100번마다 로그 출력
    if epoch % 100 == 0:
        prediction = hypothesis >= torch.FloatTensor([0.5]) # 예측값이 0.5를 넘으면 True로 간주
        correct_prediction = prediction.float() == y_train # 실제값과 일치하는 경우만 True로 간주
        accuracy = correct_prediction.sum().item() / len(correct_prediction) # 정확도를 계산
        print('Epoch {:4d}/{} Cost: {:.6f} Accuracy {:2.2f}%'.format( # 각 에포크마다 정확도를 출력
            epoch, nb_epochs, loss.item(), accuracy * 100,
        ))


In [76]:
model(x_train)

tensor([[2.8004e-04],
        [3.1763e-02],
        [3.9182e-02],
        [9.5601e-01],
        [9.9822e-01],
        [9.9968e-01]], grad_fn=<SigmoidBackward0>)

In [77]:
prediction = hypothesis >= torch.FloatTensor([0.5])
print(prediction)


tensor([[False],
        [False],
        [False],
        [ True],
        [ True],
        [ True]])


In [81]:
list(model.parameters())


[Parameter containing:
 tensor([[3.2480, 1.5152]], requires_grad=True),
 Parameter containing:
 tensor([-14.4586], requires_grad=True)]